# Avoid common logging mistakes

**Problem:** Logging configuration can be tricky, and subtle mistakes can lead to
missing messages, poor performance, or confusing behaviour.

This guide covers the most common logging mistakes and how to avoid them.

## Mistake 1: Using f-strings in log calls

This is one of the most common mistakes. F-strings are always evaluated, even
when the log message will be discarded due to the log level.

In [ ]:
import importlib
import logging

importlib.reload(logging)
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

logger = logging.getLogger("fstring_demo")

expensive_value = sum(range(1000))

# INCORRECT: f-string is always formatted, even though DEBUG is disabled
# logger.debug(f"Result: {expensive_value}")

# CORRECT: %s formatting is lazy -- only formatted if the message will be emitted
logger.debug("Result: %s", expensive_value)

print("The DEBUG message was discarded without formatting the string.")
print("With an f-string, the formatting would have happened anyway.")

## Mistake 2: Configuring logging in library code

Libraries should **never** call `logging.basicConfig()` or add handlers to loggers.
Configuration is the responsibility of the application that uses the library.

A library should only add a `logging.NullHandler` to its top-level logger to
prevent "No handlers could be found" warnings.

In [ ]:
import logging

# INCORRECT: A library calling basicConfig()
# logging.basicConfig(level=logging.DEBUG)  # Do not do this in a library

# CORRECT: A library adding a NullHandler
logger = logging.getLogger("my_library")
logger.addHandler(logging.NullHandler())

# The library uses the logger normally
logger.debug("Library debug message")
logger.info("Library info message")

print("Library messages go nowhere unless the application configures logging.")
print("This is the correct behaviour for a library.")

## Mistake 3: Swallowing exceptions

When logging exceptions, use `logger.exception()` or pass `exc_info=True` to
include the full traceback. Without it, you lose valuable debugging information.

In [ ]:
import importlib
import logging

importlib.reload(logging)
logging.basicConfig(level=logging.DEBUG, format="%(levelname)s: %(message)s")

logger = logging.getLogger("exception_demo")


def divide(a: float, b: float) -> float | None:
    """Divide a by b, handling division by zero.

    Args:
        a: The numerator.
        b: The denominator.

    Returns:
        The result, or None if division by zero.
    """
    try:
        return a / b
    except ZeroDivisionError:
        # INCORRECT: Loses the traceback
        # logger.error("Division by zero")

        # CORRECT: Includes the full traceback
        logger.exception("Division by zero")
        return None


result = divide(10, 0)
print("Result:", result)

## Mistake 4: Using the root logger directly

The module-level functions `logging.info()`, `logging.warning()`, and so on all
use the root logger. In a multi-module application, this makes it impossible to
tell which module produced a message.

In [ ]:
import importlib
import logging

importlib.reload(logging)
logging.basicConfig(
    level=logging.DEBUG,
    format="%(name)s - %(levelname)s: %(message)s"
)

# INCORRECT: Using the root logger
logging.info("Message from root logger")

# CORRECT: Using a named logger
logger = logging.getLogger("my_module")
logger.info("Message from named logger")

print("\nNotice that the named logger shows 'my_module' as the source.")

## Mistake 5: Misconfiguring log levels

A common mistake is setting the handler level but forgetting the logger level,
or the other way around. Both the logger level and the handler level must allow
a message through for it to be emitted.

In [ ]:
import logging

# INCORRECT: Handler set to DEBUG but logger defaults to WARNING
logger_bad = logging.getLogger("level_mistake")
# logger_bad level defaults to WARNING (inherited from root)
handler_bad = logging.StreamHandler()
handler_bad.setLevel(logging.DEBUG)
handler_bad.setFormatter(logging.Formatter("BAD: %(levelname)s - %(message)s"))
logger_bad.addHandler(handler_bad)

logger_bad.debug("This will NOT appear despite handler being at DEBUG")
logger_bad.warning("This will appear")

# CORRECT: Both logger and handler set to DEBUG
logger_good = logging.getLogger("level_correct")
logger_good.setLevel(logging.DEBUG)
handler_good = logging.StreamHandler()
handler_good.setLevel(logging.DEBUG)
handler_good.setFormatter(logging.Formatter("GOOD: %(levelname)s - %(message)s"))
logger_good.addHandler(handler_good)

logger_good.debug("This WILL appear")

# Clean up
logger_bad.removeHandler(handler_bad)
logger_good.removeHandler(handler_good)

## Mistake 6: Not checking for existing handlers

If you call a setup function multiple times (for example, during testing or when
reloading modules), you may end up with duplicate handlers. This causes each
message to appear multiple times.

In [ ]:
import logging


def setup_logger_bad(name: str) -> logging.Logger:
    """Set up a logger WITHOUT checking for existing handlers."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(handler)  # Adds a new handler every time
    return logger


def setup_logger_good(name: str) -> logging.Logger:
    """Set up a logger WITH a guard against duplicate handlers."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)
    if not logger.handlers:  # Only add handlers if none exist
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(message)s"))
        logger.addHandler(handler)
    return logger


# Calling setup_logger_bad twice causes duplicate output
bad_logger = setup_logger_bad("duplicate_demo")
bad_logger = setup_logger_bad("duplicate_demo")
print("Bad logger (message appears twice):")
bad_logger.info("Hello")

# Calling setup_logger_good twice is safe
good_logger = setup_logger_good("no_duplicate_demo")
good_logger = setup_logger_good("no_duplicate_demo")
print("\nGood logger (message appears once):")
good_logger.info("Hello")

# Clean up
for h in bad_logger.handlers[:]:
    bad_logger.removeHandler(h)
for h in good_logger.handlers[:]:
    good_logger.removeHandler(h)

## Mistake 7: Logging sensitive information

Be careful not to log passwords, API keys, personal data, or other sensitive
information. Log files are often stored in plain text and may be accessible
to team members or monitoring systems.

```python
# INCORRECT: Logging a password
# logger.debug("User login: username=%s, password=%s", username, password)

# CORRECT: Redact sensitive fields
# logger.debug("User login: username=%s", username)
```

## Summary checklist

| Do | Do not |
|----|--------|
| Use `%s` formatting in log calls | Use f-strings in log calls |
| Configure logging in the application entry point | Configure logging in library code |
| Use `logger.exception()` for errors | Use bare `logger.error()` in except blocks |
| Use named loggers with `logging.getLogger(__name__)` | Use the root logger directly |
| Set both logger level and handler level | Forget to set the logger level |
| Check for existing handlers before adding | Add handlers unconditionally |
| Redact sensitive data | Log passwords or API keys |